# 16 — Paper figures 2 and 3 (FIG-001)

Builds two figures for the paper from the data and the PTv3 run; no new training, no submission.

- **Fig. 2 — scanner position without labels.** (a) A test station without a tripod disc (S180): the accepted scan-line runs, the vertical planes they span (towards the scanner), the scan-line estimate and the old floor-centre estimate. (b) A validation station with a tripod disc (S163): floor points around the scanner, the disc and the scan-line estimate.
- **Fig. 3 — qualitative validation results.** Close-ups of the largest Switch, Damper and Outlet object in the validation stations: colour | ground truth | PTv3 alone | PTv3 + 2D module (the selected setting: fused D/N module, votes 3, cap 1.5).

The 2D module is the one of notebook 15 (LOSO-002), trained on the five training stations exactly as in protocol A; only the training and validation stations are processed (no test stations). The protocol-A table printed below must reproduce **HYB-006** (PTv3 seed 1: 3D only 69.94 / 51.76, selected 77.09 / 73.21).

**How to run:** Add Input → *Notebooks* → `14-ptv3-baseline` (newest version = EXP-008, seed 1). Accelerator **GPU T4 x2**, Internet **on**. Save Version → Save & Run All (about 15–20 min).

Outputs: `fig_scanner_position.pdf/.png`, `fig_qualitative_val.pdf/.png`, `summary.json`.

In [ ]:
import os, sys, json, time, subprocess, csv, glob
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy import ndimage
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE = DEV == "cpu"
CFG = dict(
    run_name="FIG-006b_figs_8pt_printed_width",
    base_run="EXP-009_PTv3_rgbi_2cm_seed2", base_name="PTv3, EXP-009",   # the 3D part (output folder of notebook 14)
    train=["S164", "S165", "S168", "S178", "S179"], val=["S163", "S169"], test=[],          # no test stations needed here
    b1_dir="/kaggle/input",                    # where the notebook-14 output is attached (searched recursively for base_run)
    feature_sets_A=[], fused_w=0.5, fused_votes=3, fused_cap=1.5,
    thr_grid=[None], min_votes_grid=[1, 2, 3], cap_grid=[None, 1.5, 2.0, 3.0], seg_voxel_m=0.02,
    fixed_variants={"hybrid_H1_mv2": ([3, 10, 14], 2, None), "hybrid_H1_mv2_cap2": ([3, 10, 14], 2, 2.0)},   # (classes, min votes, cap factor)
    final_features="D: + DINOv2 intensity (full)",
    small=[3, 5, 6, 10, 14],                                   # Damper, Exit sign, Fire terminal, Outlet, Switch
    phrases=["light switch", "electrical outlet", "power socket", "smoke detector", "exit sign", "fire extinguisher",
             "fire alarm button", "small device on a wall", "ceiling lamp", "sign", "door plate"],
    gdino="IDEA-Research/grounding-dino-base", sam="facebook/sam-vit-base", dino="facebook/dinov2-base",
    box_thr=0.15, text_thr=0.15, max_boxes=60, max_box_frac=0.08,
    views=[(y, p) for p in (-30, 0, 30) for y in range(0, 360, 45)] + [(0, 90)],
    size=1280, fov=90.0, scanner_height_m=1.6, surface_tol_m=0.02,
    sources=["rgb", "int"], merge_m=0.10, max_extent_m=0.8, min_points=10, label_frac=0.3,
    C=0.5, seed=0, dedup_iou=0.7, same_obj_iou=0.5,
    fig2_full="S180", fig2_zoom="S163",          # Fig. 2: test station without tripod disc / validation station with one
    fig3_objects=[("S163", 14, "Switch"), ("S163", 3, "Damper"), ("S169", 10, "Outlet")],   # Fig. 3 rows (largest object of the class)
    fig3_half=0.16, fig3_px=0.0025,               # Fig. 3: half width of a close-up (m) and pixel size (m)
)
if SMOKE:
    CFG.update(train=["S165"], val=["S163"], test=[], views=[(0, 0), (180, 0)], size=800, thr_grid=[None], min_votes_grid=[1, 2], cap_grid=[None, 2.0])
CLASS_NAMES = ["Column", "Components", "Covering", "Damper", "Door", "Exit sign", "Fire terminal", "Furniture",
               "Heater", "Lamp", "Outlet", "Railing", "Slab", "Stair", "Switch", "Wall", "Window", "Clutter"]
OFFSET = np.array([900.0, 480.0, 100.0])
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = os.path.join("/kaggle/working" if ON_KAGGLE else "./results", CFG["run_name"])
RAW = "/kaggle/temp/3dses" if ON_KAGGLE else "./data/3dses"
os.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
P = len(CFG["phrases"]); TEXT = " . ".join(CFG["phrases"]) + " ."
YC = [-1] + CFG["small"]                                         # classifier classes: background + the five
YNAME = {-1: "background", **{c: CLASS_NAMES[c] for c in CFG["small"]}}
FSETS = {"A: detector scores": ["gd"], "B: + 3D context": ["gd", "ctx"], "C: + DINOv2 colour": ["gd", "ctx", "er"],
         "D: + DINOv2 intensity (full)": ["gd", "ctx", "er", "ei"], "E: full without detector scores": ["ctx", "er", "ei"]}
print("device:", DEV, "| SMOKE TEST (no GPU) - numbers will mean nothing" if SMOKE else "| GPU run")
print(json.dumps({k: v for k, v in CFG.items() if k != "views"}, indent=1), "\nviews:", len(CFG["views"]))

In [ ]:
# ---- helper functions (notebooks 06 and 07, plus label-free floor / scanner position and candidate features) ----
def instances(xyz, lab, c, link=0.03, min_pts=10):
    """Split the points of class c into separate objects: points closer than `link` metres are joined.
    Returns a list of index arrays (into xyz), largest first, objects with fewer than min_pts points dropped."""
    idx = np.flatnonzero(lab == c)
    if len(idx) == 0:
        return []
    p = xyz[idx]
    pairs = cKDTree(p).query_pairs(link, output_type="ndarray")
    n = len(idx)
    if len(pairs):
        g = coo_matrix((np.ones(len(pairs), dtype=np.int8), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
        _, comp = connected_components(g, directed=False)
    else:
        comp = np.arange(n)
    out = [idx[comp == k] for k in np.unique(comp)]
    out = [o for o in out if len(o) >= min_pts]
    return sorted(out, key=len, reverse=True)


def describe(xyz, inst):
    """Size of one object: extent along its own principal axes (m), number of points, median point spacing (m)."""
    p = xyz[inst]
    q = p - p.mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    ext = np.sort(np.ptp(q @ vt.T, axis=0))[::-1]
    sp = float(np.median(cKDTree(p).query(p, k=2)[0][:, 1])) if len(p) > 1 else float("nan")
    return dict(points=int(len(p)), extent_m=[round(float(e), 3) for e in ext], spacing_m=round(sp, 4),
                centre=[round(float(v), 3) for v in p.mean(0)])


def view_frame(xyz, tree, inst, radius=0.30):
    """Camera looking straight at the surface the object sits on.
    n = normal of the surrounding surface (PCA of the neighbourhood without the object), pointing towards the
    side the object sticks out to; u, v = image axes in that surface (v as close to 'up' as possible)."""
    c = xyz[inst].mean(0)
    nb = np.asarray(tree.query_ball_point(c, radius))
    bg = np.setdiff1d(nb, inst, assume_unique=False)
    if len(bg) < 30:
        bg = nb
    q = xyz[bg] - xyz[bg].mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    n = vt[2]
    p0 = xyz[bg].mean(0)
    if (c - p0) @ n < 0:
        n = -n
    up = np.array([0.0, 0.0, 1.0])
    if abs(n @ up) > 0.9:                       # ceiling / floor: any horizontal axis will do
        up = np.array([0.0, 1.0, 0.0])
    u = np.cross(up, n); u /= np.linalg.norm(u)
    v = np.cross(n, u)
    return c, p0, n, u, v


def render_ortho(xyz, rgb, inten, lab, tree, centre, p0, n, u, v, half, px, depth=(-0.06, 0.20)):
    """Orthographic picture of everything within `half` metres of `centre` (in the image plane) and within the
    depth band in front of / behind the surface. For each pixel the point nearest to the viewer is kept.
    Returns rgb [H,W,3] uint8, intensity [H,W] float (nan = empty), label [H,W] int (-1 = empty)."""
    nb = np.asarray(tree.query_ball_point(centre, half * 1.5 + depth[1]))
    p = xyz[nb]
    d = (p - p0) @ n
    a = (p - centre) @ u
    b = (p - centre) @ v
    m = (d > depth[0]) & (d < depth[1]) & (np.abs(a) < half) & (np.abs(b) < half)
    nb, d, a, b = nb[m], d[m], a[m], b[m]
    W = int(np.ceil(2 * half / px))
    col = np.clip(((a + half) / px).astype(int), 0, W - 1)
    row = np.clip(((half - b) / px).astype(int), 0, W - 1)
    key = row * W + col
    order = np.lexsort((-d, key))               # per pixel: nearest to the viewer first
    ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((W * W, 3), 255, np.uint8); I = np.full(W * W, np.nan, np.float32); L = np.full(W * W, -1, np.int16)
    img[key[sel]] = rgb[nb[sel]]; I[key[sel]] = inten[nb[sel]]
    if lab is not None:
        L[key[sel]] = lab[nb[sel]]
    return img.reshape(W, W, 3), I.reshape(W, W), L.reshape(W, W)


def fill_small_holes(img, I, L, it=1):
    """Fill empty pixels from their neighbours (nearest filled pixel within `it` pixels) - display only."""
    empty = np.isnan(I)
    if not empty.any() or empty.all():
        return img, I, L
    dist, (ri, ci) = ndimage.distance_transform_edt(empty, return_indices=True)
    ok = empty & (dist <= it)
    img = img.copy(); I = I.copy(); L = L.copy()
    img[ok] = img[ri[ok], ci[ok]]; I[ok] = I[ri[ok], ci[ok]]; L[ok] = L[ri[ok], ci[ok]]
    return img, I, L


def estimate_scanner_xy(xyz, lab, floor_class=12, cell=0.02):
    """A terrestrial scanner leaves an unscanned disc on the floor under its tripod. Find the largest empty disc
    that is completely surrounded by floor points. Returns (x, y, radius_m, floor_z) or None."""
    f = xyz[lab == floor_class]
    if len(f) < 1000:
        return None
    lo = f[:, :2].min(0)
    g = ((f[:, :2] - lo) / cell).astype(int)
    occ = np.zeros(g.max(0) + 1, bool); occ[g[:, 0], g[:, 1]] = True
    occ = ndimage.binary_closing(occ, iterations=1)
    holes = ndimage.binary_fill_holes(occ) & ~occ
    if not holes.any():
        return None
    lab_h, nh = ndimage.label(holes)
    dist = ndimage.distance_transform_edt(holes)
    i, j = np.unravel_index(np.argmax(dist), dist.shape)
    r = float(dist[i, j] * cell)
    return float(lo[0] + (i + 0.5) * cell), float(lo[1] + (j + 0.5) * cell), r, float(np.median(f[:, 2]))


def render_panorama(xyz, rgb, inten, lab, origin, deg_per_px=0.1, max_range=30.0):
    """Equirectangular 360 x 180 degree picture seen from `origin`; per pixel the nearest point is kept."""
    q = xyz - origin
    r = np.linalg.norm(q, axis=1)
    m = (r > 0.2) & (r < max_range)
    q, r, idx = q[m], r[m], np.flatnonzero(m)
    az = np.degrees(np.arctan2(q[:, 1], q[:, 0]))          # -180..180
    el = np.degrees(np.arcsin(np.clip(q[:, 2] / r, -1, 1)))  # -90..90
    W = int(round(360 / deg_per_px)); H = int(round(180 / deg_per_px))
    col = np.clip(((az + 180) / deg_per_px).astype(int), 0, W - 1)
    row = np.clip(((90 - el) / deg_per_px).astype(int), 0, H - 1)
    key = row.astype(np.int64) * W + col
    order = np.lexsort((r, key)); ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    img = np.full((H * W, 3), 255, np.uint8); I = np.full(H * W, np.nan, np.float32); L = np.full(H * W, -1, np.int16)
    img[key[sel]] = rgb[idx[sel]]; I[key[sel]] = inten[idx[sel]]
    if lab is not None:
        L[key[sel]] = lab[idx[sel]]
    filled = len(sel) / (H * W)
    return img.reshape(H, W, 3), I.reshape(H, W), L.reshape(H, W), filled


def stretch(I, lo=2, hi=98):
    """Intensity -> 0..255 grey with a percentile stretch (empty pixels white)."""
    v = I[~np.isnan(I)]
    if len(v) == 0:
        return np.full(I.shape, 255, np.uint8)
    a, b = np.percentile(v, [lo, hi]); b = max(b, a + 1e-6)
    g = np.clip((I - a) / (b - a), 0, 1) * 255
    g[np.isnan(I)] = 255
    return g.astype(np.uint8)


def view_dirs(yaw_deg, pitch_deg):
    """Camera axes for a view: f = looking direction, r = image right, d = image down (all unit vectors)."""
    y, p = np.radians(yaw_deg), np.radians(pitch_deg)
    f = np.array([np.cos(p) * np.cos(y), np.cos(p) * np.sin(y), np.sin(p)])
    up = np.array([0.0, 0.0, 1.0]) if abs(pitch_deg) < 80 else np.array([np.cos(y), np.sin(y), 0.0])
    r = np.cross(f, up); r /= np.linalg.norm(r)
    d = np.cross(f, r)                                   # points "down" in the picture
    if d @ up > 0: d = -d
    return f, r, d


def render_view(xyz, rgb, inten, origin, yaw, pitch, size=1280, fov_deg=90.0, fill_px=3, leak=0.06):
    """Pinhole picture from `origin`. For every pixel the nearest point is kept; points that shine through gaps in a
    nearer surface are removed (depth much larger than the local minimum), then small holes are filled from the
    nearest filled pixel. Returns rgb [S,S,3] uint8, intensity [S,S] float32 (nan = empty), idx [S,S] int64
    (index of the 3D point shown in the pixel, -1 = empty)."""
    f, r, d = view_dirs(yaw, pitch)
    q = xyz - origin
    z = q @ f
    m = z > 0.15
    zi = z[m]; qi = q[m]; ii = np.flatnonzero(m)
    fpx = (size / 2) / np.tan(np.radians(fov_deg) / 2)
    u = fpx * (qi @ r) / zi + size / 2
    v = fpx * (qi @ d) / zi + size / 2
    ok = (u >= 0) & (u < size) & (v >= 0) & (v < size)
    u, v, zi, ii = u[ok].astype(np.int64), v[ok].astype(np.int64), zi[ok], ii[ok]
    key = v * size + u
    order = np.lexsort((zi, key)); ks = key[order]
    first = np.ones(len(order), bool); first[1:] = ks[1:] != ks[:-1]
    sel = order[first]
    Z = np.full(size * size, np.inf, np.float32); IDX = np.full(size * size, -1, np.int64)
    Z[key[sel]] = zi[sel]; IDX[key[sel]] = ii[sel]
    Z = Z.reshape(size, size); IDX = IDX.reshape(size, size)
    zmin = ndimage.minimum_filter(Z, size=5, mode="nearest")
    bad = np.isfinite(Z) & (Z > zmin * (1 + leak) + 0.05)
    Z[bad] = np.inf; IDX[bad] = -1
    empty = IDX < 0
    if empty.any() and (~empty).any():
        dist, (ri, ci) = ndimage.distance_transform_edt(empty, return_indices=True)
        fill = empty & (dist <= fill_px)
        IDX[fill] = IDX[ri[fill], ci[fill]]; Z[fill] = Z[ri[fill], ci[fill]]
    img = np.full((size, size, 3), 255, np.uint8); I = np.full((size, size), np.nan, np.float32)
    has = IDX >= 0
    img[has] = rgb[IDX[has]]; I[has] = inten[IDX[has]]
    proj = (u, v, zi, ii, Z)              # every point that falls in the picture, and the front depth per pixel
    return img, I, IDX, proj


def mask_to_points_full(mask, proj, tol=0.03):
    """All 3D points that project into the mask and lie on the visible surface (depth within `tol` of the front
    point of their pixel) - not only the one point per pixel that the picture shows."""
    u, v, zi, ii, Z = proj
    inside = mask[v, u]
    zf = Z[v[inside], u[inside]]
    keep = zi[inside] <= zf + tol
    return np.unique(ii[inside][keep])


def mask_to_points_full(mask, proj, tol=0.03):
    """All 3D points that project into the mask and lie on the visible surface (depth within `tol` of the front
    point of their pixel) - not only the one point per pixel that the picture shows."""
    u, v, zi, ii, Z = proj
    inside = mask[v, u]
    zf = Z[v[inside], u[inside]]
    keep = zi[inside] <= zf + tol
    return np.unique(ii[inside][keep])


def phrase_to_class(phrase, prompts):
    """Map a phrase returned by the detector to a class ID by the words of the prompts (longest match wins)."""
    p = phrase.lower(); best, best_len = None, 0
    for c, plist in prompts.items():
        for pr in plist:
            for w in pr.lower().split():
                if len(w) > 3 and w in p and len(w) > best_len:
                    best, best_len = c, len(w)
    return best


def mask_to_points(mask, IDX):
    """3D point indices behind the pixels of a 2D mask."""
    ids = IDX[mask]
    return np.unique(ids[ids >= 0])




def floor_ceiling(z, step=0.02):
    """Floor and ceiling height without labels: the most populated 2 cm height bin in the lowest 30 % of the height
    range (floor) and between 2.0 and 4.5 m above it (ceiling) - floors and ceilings are large horizontal surfaces."""
    lo, hi = float(np.percentile(z, 0.1)), float(np.percentile(z, 99.9))
    edges = lo - step + step * np.arange(int((hi - lo) / step) + 4)      # one spare bin below and two above
    h, _ = np.histogram(z, bins=edges)
    c = 0.5 * (edges[:-1] + edges[1:])
    low = c < lo + 0.3 * (hi - lo)
    floor = float(c[low][np.argmax(h[low])])
    high = (c > floor + 2.0) & (c < floor + 4.5)                    # ceiling: 2.0-4.5 m above the floor (ignores stairwells)
    if not high.any(): high = c > hi - 0.3 * (hi - lo)
    return floor, float(c[high][np.argmax(h[high])])


def scanner_position(xyz, floor_z, height, cell=0.02):
    """Scanner (x, y) from the unscanned disc on the floor under the tripod, found without labels: floor points =
    within 3 cm of the floor height. Falls back to the centre of the floor. Returns (origin, how)."""
    fm = np.abs(xyz[:, 2] - floor_z) < 0.03
    lab = np.where(fm, 12, -1)
    e = estimate_scanner_xy(xyz, lab, 12, cell)
    if e is not None and e[2] >= 0.15:
        return np.array([e[0], e[1], floor_z + height]), f"tripod disc r={e[2]:.2f} m"
    f = xyz[fm]
    return np.array([np.median(f[:, 0]), np.median(f[:, 1]), floor_z + height]), "floor centre (no tripod disc)"


def phrase_index(label, phrases):
    """Index of the proposal phrase a detector label belongs to (exact match, else most shared words)."""
    l = str(label).lower().strip()
    if l in phrases:
        return phrases.index(l)
    lw = set(l.split()); best, bi = 0, None
    for i, p in enumerate(phrases):
        k = len(lw & set(p.split()))
        if k > best: best, bi = k, i
    return bi


def cluster_centroids(C, r):
    """Group points (proposal centroids) closer than r; returns a component id per point."""
    if len(C) == 0:
        return np.zeros(0, int)
    pairs = cKDTree(C).query_pairs(r, output_type="ndarray")
    n = len(C)
    g = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n)) if len(pairs) else coo_matrix((n, n))
    return connected_components(g, directed=False)[1]


def context_features(xyz, rgb, inten, tree, ids, floor_z, ceil_z, origin, radius=0.30):
    """3D context of one candidate (label-free): height above floor / below ceiling, orientation of the surface it
    sits on, size, how far it sticks out, colour and intensity and their contrast to the surroundings, range."""
    p = xyz[ids]; c = p.mean(0)
    nb = np.asarray(tree.query_ball_point(c, radius))
    bg = np.setdiff1d(nb, ids)
    if len(bg) < 30: bg = nb
    q = xyz[bg] - xyz[bg].mean(0)
    _, _, vt = np.linalg.svd(q, full_matrices=False); n = vt[2]; p0 = xyz[bg].mean(0)
    if (origin - p0) @ n < 0: n = -n
    prot = (p - p0) @ n
    qq = p - c
    ext = np.sort(np.ptp(qq @ np.linalg.svd(qq, full_matrices=False)[2].T, axis=0))[::-1] if len(p) > 3 else np.zeros(3)
    ib, ic = float(inten[bg].mean()), float(inten[ids].mean())
    cb, cc = rgb[bg].astype(float).mean(0), rgb[ids].astype(float).mean(0)
    return np.array([c[2] - floor_z, ceil_z - c[2], abs(n[2]), ext[0], ext[1], ext[2], np.log1p(len(ids)),
                     np.median(prot), np.percentile(prot, 95), ic, ib, ic - ib, *(cc / 255), *((cc - cb) / 255),
                     float(rgb[ids].astype(float).std() / 255), float(np.linalg.norm(c - origin))], np.float32)

CTX_NAMES = ["h_above_floor", "h_below_ceiling", "abs_normal_z", "ext0", "ext1", "ext2", "log_points", "protrusion_med",
             "protrusion_p95", "int_obj", "int_bg", "int_contrast", "r", "g", "b", "dr", "dg", "db", "rgb_std", "range"]


def scan_order_test(xyz, n=200000):
    """Is the file in scanner order? Median distance between consecutive points (m)."""
    d = np.linalg.norm(np.diff(xyz[:n], axis=0), axis=1)
    return float(np.median(d))


def scanner_by_scanlines(xyz, chunk=100, min_xy=0.15, max_res=0.02, iters=3):
    """If points are stored in scan order, every vertical scan line lies in a vertical plane through the scanner:
    intersect the xy lines of many short runs of consecutive points (least squares, outliers removed)."""
    N, C = [], []
    for s in range(0, len(xyz) - chunk, chunk):
        p = xyz[s:s + chunk]
        if np.ptp(p[:, :2], axis=0).max() < min_xy: continue                       # needs radial movement (floor / ceiling)
        if np.linalg.norm(np.diff(p, axis=0), axis=1).max() > 0.5: continue        # run crosses a jump
        q = p[:, :2] - p[:, :2].mean(0)
        u, sv, vt = np.linalg.svd(q, full_matrices=False)
        if sv[1] / max(sv[0], 1e-9) > 0.05 or np.sqrt(sv[1] ** 2 / chunk) > max_res: continue
        d = vt[0]; N.append([-d[1], d[0]]); C.append(p[:, :2].mean(0))
    if len(N) < 20: return None, 0
    N, C = np.array(N), np.array(C); keep = np.ones(len(N), bool)
    for _ in range(iters):
        A = N[keep]; b = (N[keep] * C[keep]).sum(1)
        x = np.linalg.lstsq(A, b, rcond=None)[0]
        res = np.abs((N * (x - C)).sum(1)); keep = res < max(0.05, np.percentile(res[keep], 80))
    return x, dict(lines=int(keep.sum()), of=len(N), median_residual_m=float(np.median(res[keep])))




def scanner_position_v4(xyz, floor_z, height):
    # FEAS-005: the files keep scanner order -> intersect the vertical planes of short runs of consecutive points
    # (matches the tripod disc within 1.6-4.2 cm); falls back to the tripod disc / floor centre otherwise
    if scan_order_test(xyz) < 0.1:
        x, info = scanner_by_scanlines(xyz)
        if x is not None and info["median_residual_m"] < 0.03:
            return np.array([x[0], x[1], floor_z + height]), f"scan lines ({info['lines']} lines)"
    return scanner_position(xyz, floor_z, height)

In [ ]:
# ---- self-test on a synthetic room (label-free floor and scanner position, merging, context features) ----
def _selftest():
    s = 0.01; Pp, Lb = [], []
    def grid(a0, a1, b0, b1):
        A, B = np.meshgrid(np.arange(a0, a1, s), np.arange(b0, b1, s), indexing="ij"); return A.ravel(), B.ravel()
    x, y = grid(0, 4, 0, 4); k = (x - 1.5) ** 2 + (y - 2.0) ** 2 > 0.3 ** 2
    Pp.append(np.c_[x[k], y[k], np.zeros(k.sum())]); Lb.append(np.full(k.sum(), 12))
    x, y = grid(0, 4, 0, 4); Pp.append(np.c_[x, y, np.full(len(x), 3.0)]); Lb.append(np.full(len(x), 2))
    for X in (0.0, 4.0):
        y, z = grid(0, 4, 0, 3); Pp.append(np.c_[np.full(len(y), X), y, z]); Lb.append(np.full(len(y), 15))
    yy, zz = np.meshgrid(np.arange(0.96, 1.04, 0.003), np.arange(1.06, 1.14, 0.003), indexing="ij")
    Pp.append(np.c_[np.full(yy.size, 0.015), yy.ravel(), zz.ravel()]); Lb.append(np.full(yy.size, 14))
    xyz = np.concatenate(Pp) + np.array([10, 20, 5.0]); lab = np.concatenate(Lb)
    rgb = np.full((len(xyz), 3), 200, np.uint8); inten = np.full(len(xyz), 0.3, np.float32); inten[lab == 14] = 0.8
    fz, cz = floor_ceiling(xyz[:, 2]); assert abs(fz - 5.0) < 0.02 and abs(cz - 8.0) < 0.02, (fz, cz)
    o, how = scanner_position(xyz, fz, 1.6); assert abs(o[0] - 11.5) < 0.03 and abs(o[1] - 22.0) < 0.03, (o, how)
    ids = np.flatnonzero(lab == 14); f = context_features(xyz, rgb, inten, cKDTree(xyz), ids, fz, cz, o)
    assert abs(f[0] - 1.10) < 0.02 and f[2] < 0.1 and f[11] > 0.4 and 0.012 < f[7] < 0.018, f[:12]
    assert phrase_index("light switch", CFG["phrases"]) == 0 and phrase_index("lamp", CFG["phrases"]) == 8
    comp = cluster_centroids(np.array([[0, 0, 0], [0.05, 0, 0], [1, 1, 1.0]]), 0.1); assert comp[0] == comp[1] != comp[2]
    print("self-test passed: floor", round(fz, 3), "ceiling", round(cz, 3), "|", how, "| switch height", round(float(f[0]), 3))
_selftest()

In [ ]:
# ---- models: Grounding DINO (proposals), SAM (masks), DINOv2 (crop embeddings) ----
import transformers
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor, AutoImageProcessor, AutoModel
print("transformers", transformers.__version__, "| torch", torch.__version__)
t0 = time.time()
gd_proc = AutoProcessor.from_pretrained(CFG["gdino"]); gd = AutoModelForZeroShotObjectDetection.from_pretrained(CFG["gdino"]).to(DEV).eval()
sam_proc = SamProcessor.from_pretrained(CFG["sam"]); sam = SamModel.from_pretrained(CFG["sam"]).to(DEV).eval()
dn_proc = AutoImageProcessor.from_pretrained(CFG["dino"]); dn = AutoModel.from_pretrained(CFG["dino"]).to(DEV).eval()
print(f"models loaded in {time.time() - t0:.0f} s | prompt: {TEXT}")

def detect(pil):
    # Grounding DINO -> list of (box, score, phrase index); class-agnostic use: every phrase is kept
    inp = gd_proc(images=pil, text=TEXT, return_tensors="pt").to(DEV)
    with torch.no_grad():
        out = gd(**inp)
    kw = dict(text_threshold=CFG["text_thr"], target_sizes=[pil.size[::-1]])
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=CFG["box_thr"], **kw)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=CFG["box_thr"], **kw)[0]
    labels = r.get("text_labels", r.get("labels"))
    W, H = pil.size; dets = []
    for b, s_, l in zip(r["boxes"].cpu().numpy(), r["scores"].cpu().numpy(), labels):
        pi = phrase_index(l, CFG["phrases"])
        area = max(b[2] - b[0], 0) * max(b[3] - b[1], 0)
        if pi is None or area < 16 or area > CFG["max_box_frac"] * W * H: continue
        dets.append((b.astype(float), float(s_), pi))
    dets.sort(key=lambda d: -d[1])
    return dets[:CFG["max_boxes"]]

def segment(pil, boxes):
    # SAM: best of three masks per box, clipped to the box + 10 %
    res = []
    for i0 in range(0, len(boxes), 32):
        bb = boxes[i0:i0 + 32]
        inp = sam_proc(pil, input_boxes=[[list(map(float, b)) for b in bb]], return_tensors="pt").to(DEV)
        with torch.no_grad():
            out = sam(**inp, multimask_output=True)
        masks = sam_proc.image_processor.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0]
        iou = out.iou_scores[0].cpu().numpy()
        for i, b in enumerate(bb):
            m = masks[i, int(iou[i].argmax())].numpy().astype(bool)
            w, h = b[2] - b[0], b[3] - b[1]; clip = np.zeros_like(m)
            clip[int(max(b[1] - 0.1 * h, 0)):int(b[3] + 0.1 * h) + 1, int(max(b[0] - 0.1 * w, 0)):int(b[2] + 0.1 * w) + 1] = True
            m &= clip
            if m.sum() < 4:
                m = np.zeros_like(m); m[int(b[1]):int(b[3]) + 1, int(b[0]):int(b[2]) + 1] = True
            res.append(m)
    return res

def embed(crops):
    # DINOv2 image embedding (pooled CLS token), L2-normalised
    out = []
    for i0 in range(0, len(crops), 32):
        inp = dn_proc(images=crops[i0:i0 + 32], return_tensors="pt").to(DEV)
        with torch.no_grad():
            o = dn(**inp)
        e = o.pooler_output if getattr(o, "pooler_output", None) is not None else o.last_hidden_state[:, 0]
        out.append(F.normalize(e.float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 768), np.float32)

def crop(pil, b, pad=0.25):
    w, h = b[2] - b[0], b[3] - b[1]; p = pad * max(w, h) + 4
    return pil.crop((int(max(b[0] - p, 0)), int(max(b[1] - p, 0)), int(b[2] + p), int(b[3] + p)))

In [ ]:
# ---- per station: views -> proposals with features (labelled stations: also labels and one example per object) ----
def raw_file(sub, s):
    rel = os.path.join(sub, s + ".npy")
    if os.path.isfile(os.path.join(RAW, rel)): return os.path.join(RAW, rel)
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download
    snapshot_download(repo_id="heig-vd-geo/3DSES", repo_type="dataset", allow_patterns=[sub + "/" + s + ".npy"], local_dir=RAW)
    return os.path.join(RAW, rel)

def box_iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(x1 - x0, 0) * max(y1 - y0, 0)
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0.0

def merge_dets(dets):
    dets = sorted(dets, key=lambda d: -d[1]); kept = []
    for b, sc, pi in dets:
        if all(box_iou(b, k[0]) < CFG["dedup_iou"] for k in kept):
            kept.append([b, sc, np.zeros(P, np.float32)])
    for b, sc, pi in dets:
        for k in kept:
            if box_iou(b, k[0]) >= CFG["same_obj_iou"]: k[2][pi] = max(k[2][pi], sc)
    return kept

def proposal_label(ids, lab, pid, gsize, gcls):
    fr = {c: float((lab[ids] == c).mean()) for c in CFG["small"]}
    cb = max(fr, key=fr.get)
    if fr[cb] >= CFG["label_frac"]: return cb
    k = pid[ids]; k = k[k >= 0]
    if len(k):
        kk, n = np.unique(k, return_counts=True)
        for kq, nq in zip(kk, n):
            c = gcls[kq]
            if nq >= 0.5 * gsize[kq] and fr[c] >= 0.1: return c
    return -1

def process_station(s, sub):
    t0 = time.time()
    a = np.load(raw_file(sub, s))
    xyz = a[:, :3] - OFFSET; rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8)
    inten = a[:, 6].astype(np.float32); lab = a[:, 7].astype(np.int64) if sub == "Gold" else None
    del a
    fz, cz = floor_ceiling(xyz[:, 2]); origin, how = scanner_position_v4(xyz, fz, CFG["scanner_height_m"])
    tree = cKDTree(xyz)
    gobj = [(c, o) for c in CFG["small"] for o in instances(xyz, lab, c, 0.03, CFG["min_points"])] if lab is not None else []
    pid = np.full(len(xyz), -1, np.int64)
    for k, (c, o) in enumerate(gobj): pid[o] = k
    gsize = np.array([len(o) for c, o in gobj]); gcls = [c for c, o in gobj]
    best = {k: [0, None, None, None] for k in range(len(gobj))}
    gd_gt = np.zeros((len(gobj), P), np.float32)
    props = []
    for vi, (yw, pt) in enumerate(CFG["views"]):
        img, I, IDX, proj = render_view(xyz, rgb, inten, origin, yw, pt, size=CFG["size"], fov_deg=CFG["fov"])
        g = stretch(I); pics = {"rgb": Image.fromarray(img), "int": Image.fromarray(np.dstack([g, g, g]))}
        vis_boxes = {}
        if gobj:
            ov = np.where(IDX >= 0, pid[np.maximum(IDX, 0)], -1)
            for k in np.unique(ov[ov >= 0]):
                ys, xs = np.nonzero(ov == k); b = np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1], float)
                vis_boxes[int(k)] = b
                if len(xs) > best[int(k)][0]:
                    best[int(k)] = [len(xs), crop(pics["rgb"], b), crop(pics["int"], b), b]
        raw = [d for src in CFG["sources"] for d in detect(pics[src])]
        if not raw: continue
        for (b, sc, pi) in raw:
            for k, vb in vis_boxes.items():
                if box_iou(b, vb) > 0.1: gd_gt[k, pi] = max(gd_gt[k, pi], sc)
        dets = merge_dets(raw)
        masks = segment(pics["rgb"], [d[0] for d in dets])
        er = embed([crop(pics["rgb"], d[0]) for d in dets]); ei = embed([crop(pics["int"], d[0]) for d in dets])
        for j, ((b, sc, gdv), m) in enumerate(zip(dets, masks)):
            ids = mask_to_points_full(m, proj, CFG["surface_tol_m"])
            if len(ids) < CFG["min_points"] or np.ptp(xyz[ids], axis=0).max() > CFG["max_extent_m"]: continue
            props.append(dict(ids=ids.astype(np.int32), y=proposal_label(ids, lab, pid, gsize, gcls) if lab is not None else -1, view=vi,
                              gd=np.r_[gdv, gdv.max()].astype(np.float32), ctx=context_features(xyz, rgb, inten, tree, ids, fz, cz, origin),
                              er=er[j], ei=ei[j]))
    gtc = []
    ks = [k for k in range(len(gobj)) if best[k][1] is not None]
    if ks:
        er = embed([best[k][1] for k in ks]); ei = embed([best[k][2] for k in ks])
        for j, k in enumerate(ks):
            c, o = gobj[k]
            gtc.append(dict(y=c, gd=np.r_[gd_gt[k], gd_gt[k].max()].astype(np.float32),
                            ctx=context_features(xyz, rgb, inten, tree, o, fz, cz, origin), er=er[j], ei=ei[j], k=k))
    covered = np.zeros(len(xyz), bool)
    for p_ in props: covered[p_["ids"]] = True
    S = dict(n=len(xyz), lab=lab.astype(np.int8) if lab is not None else None, gobj=gobj, props=props, gtc=gtc,
             prop_recall=[float(covered[o].mean()) >= 0.3 for (c, o) in gobj], how=how, floor=fz, ceiling=cz,
             seconds=round(time.time() - t0, 1), xyz=xyz.astype(np.float32))
    npos = {YNAME[c]: sum(p_["y"] == c for p_ in props) for c in CFG["small"]} if lab is not None else "test station (no labels)"
    print(f"{s} ({sub}): {how}; floor {fz:.2f} ceiling {cz:.2f}; {len(props)} proposals; positives {npos}; "
          + (f"objects covered {sum(S['prop_recall'])}/{len(gobj)}; " if gobj else "") + f"{time.time() - t0:.0f} s")
    return S

STN = {}
t_all = time.time()
for s in CFG["train"] + CFG["val"]:
    STN[s] = process_station(s, "Gold")
for s in CFG["test"]:
    STN[s] = process_station(s, "test_area")
for S in STN.values():      # colour-free 3D context for set N: drop r, g, b, dr, dg, db, rgb_std (indices 12-18)
    for it in S["props"] + S["gtc"]: it["ctxnc"] = np.delete(it["ctx"], list(range(12, 19))).astype(np.float32)
print(f"all stations in {(time.time() - t_all) / 60:.1f} min")

In [ ]:
# ---- 2D classifier, 3D predictions of B1 (EXP-003), and their combination ----
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA

def features(train, test, parts, k=32):
    Xtr, Xte = [], []
    for p_ in parts:
        A = np.array([it[p_] for it in train], np.float32); B = np.array([it[p_] for it in test], np.float32)
        if p_ in ("er", "ei"):
            pca = PCA(n_components=min(k, len(A) - 1), random_state=CFG["seed"]).fit(A)
            A, B = pca.transform(A), pca.transform(B)
        Xtr.append(A); Xte.append(B)
    return np.concatenate(Xtr, 1), np.concatenate(Xte, 1)

def train_predict(train_st, test_st, parts):
    # fit on the proposals + per-object examples of train_st; per test station: per-point mean class probabilities
    train = [it for t in train_st for it in STN[t]["props"] + STN[t]["gtc"]]
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=CFG["C"], class_weight="balanced", max_iter=5000, random_state=CFG["seed"]))
    out = {}
    tests = [STN[s]["props"] for s in test_st]
    allt = [it for L in tests for it in L]
    Xtr, Xte = features(train, allt, parts)
    clf.fit(Xtr, np.array([it["y"] for it in train]))
    pr = clf.predict_proba(Xte) if len(allt) else np.zeros((0, len(clf.classes_)))
    i0 = 0
    for s, L in zip(test_st, tests):
        p_s = pr[i0:i0 + len(L)]; i0 += len(L)
        n = STN[s]["n"]; acc = np.zeros((n, len(clf.classes_)), np.float32); cnt = np.zeros(n, np.int16)
        for it, p in zip(L, p_s):
            acc[it["ids"]] += p; cnt[it["ids"]] += 1
        has = cnt > 0
        acc[has] /= cnt[has][:, None]
        out[s] = dict(mean=acc, cnt=cnt, classes=list(clf.classes_))
    return out

def small_override(base, P2, classes_on, thr=None, min_votes=1):
    # 3D prediction, overwritten by the 2D module where it predicts one of classes_on
    cls = np.array(P2["classes"]); best = P2["mean"].argmax(1); bestp = P2["mean"].max(1)
    lab2 = cls[best]
    ok = (P2["cnt"] >= min_votes) & np.isin(lab2, classes_on)
    if thr is not None: ok &= bestp >= thr
    out = base.copy(); out[ok] = lab2[ok]
    return out

def object_caps(stations, factor):
    # cap per small class = factor x the largest labelled object of that class in `stations` (points); None = no cap
    if factor is None: return {}
    caps = {}
    for t in stations:
        for c, o in STN[t]["gobj"]: caps[c] = max(caps.get(c, 0), len(o))
    return {c: int(factor * v) for c, v in caps.items()}

def segments(xyz, idx, vox):
    # connected segments of the points idx: 2 cm voxels, 26-neighbourhood; returns a segment id per point of idx
    v = np.floor(xyz[idx] / vox).astype(np.int64)
    uv, inv = np.unique(v, axis=0, return_inverse=True); inv = inv.ravel()
    pairs = cKDTree(uv).query_pairs(1.75, output_type="ndarray")
    if len(pairs):
        g = coo_matrix((np.ones(len(pairs), np.int8), (pairs[:, 0], pairs[:, 1])), shape=(len(uv), len(uv)))
        comp = connected_components(g, directed=False)[1]
    else:
        comp = np.arange(len(uv))
    return comp[inv]

def cap_segments(base, out, xyz, caps, vox, stats=None):
    # points changed by the 2D module to class c, in segments larger than caps[c], go back to the 3D label
    out = out.copy()
    for c, cap in caps.items():
        idx = np.flatnonzero((out == c) & (base != c))
        if len(idx) == 0: continue
        seg = segments(xyz, idx, vox); n = np.bincount(seg)
        big = n[seg] > cap
        out[idx[big]] = base[idx[big]]
        if stats is not None:
            stats[CLASS_NAMES[c]] = dict(cap=cap, changed_points=int(len(idx)), segments=int(len(n)), largest=int(n.max()),
                                         removed_segments=int((n > cap).sum()), removed_points=int(big.sum()))
    return out

def hybrid(base, P2, classes_on, mv, factor, xyz, cap_stations, stats=None):
    out = small_override(base, P2, classes_on, None, mv)
    caps = {c: v for c, v in object_caps(cap_stations, factor).items() if c in classes_on}
    return cap_segments(base, out, xyz, caps, CFG["seg_voxel_m"], stats) if caps else out

def _selftest_cap():
    xyz = np.r_[np.random.RandomState(0).rand(500, 3) * 0.05, np.random.RandomState(1).rand(20000, 3) * [1, 1, 0.01] + [5, 5, 0]]
    base = np.zeros(len(xyz), np.int64); out = np.full(len(xyz), 3)
    r = cap_segments(base, out, xyz, {3: 1000}, 0.02)
    assert (r[:500] == 3).all() and (r[500:] == 0).all()
    print("segment-size filter self-test passed")
_selftest_cap()

def find_b1():
    hits = glob.glob(os.path.join(CFG["b1_dir"], "**", CFG["base_run"], "pred_val_2cm_S163.npy"), recursive=True)
    if not hits:
        print("!" * 100 + "\nThe output of notebook 14 (" + CFG["base_run"] + ") is not attached (Add Input -> Notebooks -> 14-ptv3-baseline). The 3D part and the "
              "combination are skipped" + (" (fine for the smoke test)." if SMOKE else " - THIS RUN WILL NOT PRODUCE THE MAIN RESULT.") + "\n" + "!" * 100)
        return None
    return os.path.dirname(hits[0])

def voxel_subsample_inv(xyz, size):
    g = np.floor(xyz / size).astype(np.int64)
    d2 = ((xyz - (g + 0.5) * size) ** 2).sum(1)
    g -= g.min(0)
    key = g[:, 0] | (g[:, 1] << 20) | (g[:, 2] << 40)
    order = np.lexsort((d2, key)); ks = key[order]
    first = np.ones(len(order), dtype=bool); first[1:] = ks[1:] != ks[:-1]
    inv = np.empty(len(order), dtype=np.int64); inv[order] = np.cumsum(first) - 1
    return order[first], inv

def confusion(gt, pred, C=18):
    return np.bincount(gt * C + pred, minlength=C * C).reshape(C, C)

def metrics(cm):
    tp = np.diag(cm).astype(float); gt = cm.sum(1); pr = cm.sum(0); present = gt > 0
    iou = np.where(present, tp / np.maximum(gt + pr - tp, 1), np.nan) * 100
    sm = [c for c in CFG["small"] if present[c]]
    return dict(mIoU=round(float(np.nanmean(iou)), 2), small_mIoU=round(float(np.mean([iou[c] for c in sm])), 2),
                OA=round(float(tp.sum() / cm.sum() * 100), 2), iou={CLASS_NAMES[c]: round(float(iou[c]), 2) for c in range(18) if present[c]})

B1 = find_b1()
if B1: print(CFG["base_run"] + " outputs:", B1, sorted(os.listdir(B1)))

In [ ]:
# ---- the 2D module of LOSO-002 and the hybrid ----
SET_D = ["gd", "ctx", "er", "ei"]; SET_N = ["gd", "ctxnc"]
def hybrid_module(base, PD, PN, mv, factor, xyz, cap_stations, fused=True, w=None, stats=None, damper=True):
    # Damper from D; Switch / Outlet from (1-w) D + w N (fused) or from D alone; votes mv; segment cap factor
    w = CFG["fused_w"] if w is None else w; n = len(base); empty = np.full(n, -1)
    dam = small_override(empty, PD, [3], None, mv)
    if fused:
        assert PD["classes"] == PN["classes"]
        P = dict(mean=(1 - w) * PD["mean"] + w * PN["mean"], cnt=PD["cnt"], classes=PD["classes"])
    else:
        P = PD
    so = small_override(empty, P, [10, 14], None, mv)
    out = base.copy()
    if damper: out[dam == 3] = 3                     # damper=False: the 3D network keeps its own Damper
    m = np.isin(so, [10, 14]); out[m] = so[m]
    caps = {c: v for c, v in object_caps(cap_stations, factor).items() if c in ((3, 10, 14) if damper else (10, 14))}
    return cap_segments(base, out, xyz, caps, CFG["seg_voxel_m"], stats) if caps else out

# ---- PROTOCOL A: 2D module trained on the 5 training stations, S163 + S169, original points ----
SUMMARY = {"cfg": {k: v for k, v in CFG.items() if k != "views"}, "smoke": SMOKE,
           "stations": {s: dict(how=S["how"], floor=round(S["floor"], 3), ceiling=round(S["ceiling"], 3), proposals=len(S["props"]),
                                seconds=S["seconds"]) for s, S in STN.items()}}
base3d = {}
for s in (CFG["val"] if B1 else []):
    keep, inv = voxel_subsample_inv(STN[s]["xyz"].astype(np.float64), 0.02)
    pv = np.load(os.path.join(B1, f"pred_val_2cm_{s}.npy")).astype(np.int64)
    assert len(pv) == len(keep), (s, len(pv), len(keep))
    base3d[s] = pv[inv]
rows = []
if B1:
    rows.append(("3D only (" + CFG["base_name"] + ")", None, None, metrics(sum(confusion(STN[s]["lab"].astype(np.int64), base3d[s]) for s in CFG["val"]))))
    PDv = train_predict(CFG["train"], CFG["val"], SET_D); PNv = train_predict(CFG["train"], CFG["val"], SET_N)
    for name, damper in (("H1 fused (LOSO-002 module, 2D Damper)", True), ("Switch/Outlet only (3D keeps Damper)", False)):
        for mv in (2, 3):
            for cf in (None, 1.5):
                cm = sum(confusion(STN[s]["lab"].astype(np.int64), hybrid_module(base3d[s], PDv[s], PNv[s], mv, cf, STN[s]["xyz"], CFG["train"], True, damper=damper))
                         for s in CFG["val"])
                rows.append((name, mv, cf, metrics(cm)))
print(f"{'variant':<44} {'votes':>5} {'cap':>5} | {'mIoU':>6} {'small':>6} | " + " ".join(f"{CLASS_NAMES[c][:7]:>7}" for c in CFG["small"]))
for v, mv, cf, m in rows:
    print(f"{v:<44} {str(mv):>5} {str(cf):>5} | {m['mIoU']:>6} {m['small_mIoU']:>6} | " + " ".join(f"{m['iou'].get(CLASS_NAMES[c], float('nan')):>7}" for c in CFG["small"]))
SUMMARY["protocol_A"] = [dict(variant=v, min_votes=mv, cap_factor=cf, **m) for v, mv, cf, m in rows]
with open(os.path.join(OUT, "summary.json"), "w") as fh: json.dump(SUMMARY, fh, indent=1)
# pre-registered choice (fixed before the run): votes 3, cap 1.5 from LOSO-002; damper or not by validation mIoU
_cand = [(v, m) for v, mv, cf, m in rows if mv == CFG["fused_votes"] and cf == CFG["fused_cap"]]
if _cand:
    _best = max(_cand, key=lambda t: t[1]["mIoU"])
    SUMMARY["validation_selected"] = dict(variant=_best[0], mIoU=_best[1]["mIoU"], small_mIoU=_best[1]["small_mIoU"],
                                          zip="submission_hybrid_ptv3_fused.zip" if "2D Damper" in _best[0] else "submission_hybrid_ptv3_fused_so.zip")
    print("\nvalidation-selected (votes %d, cap %s):" % (CFG["fused_votes"], CFG["fused_cap"]), SUMMARY["validation_selected"])
    with open(os.path.join(OUT, "summary.json"), "w") as fh: json.dump(SUMMARY, fh, indent=1)


In [ ]:
# ---- FIGURES: Fig. 2 (scanner position) and Fig. 3 (qualitative, validation) ----
# Self-contained helpers; they use only numpy / scipy / matplotlib and the functions of the cells above
# (floor_ceiling, scanner_position, estimate_scanner_xy).
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Patch
from scipy import ndimage
from scipy.spatial import cKDTree

plt.rcParams.update({"font.family": "STIXGeneral", "font.size": 8, "pdf.fonttype": 42, "mathtext.fontset": "stix"})

def scanline_runs(xyz, chunk=100, min_xy=0.15, max_res=0.02, iters=3):
    """Same test as scanner_by_scanlines, but also returns the accepted runs (centre, direction, xy extent) for plotting."""
    N, C, E = [], [], []
    for s in range(0, len(xyz) - chunk, chunk):
        p = xyz[s:s + chunk]
        if np.ptp(p[:, :2], axis=0).max() < min_xy: continue
        if np.linalg.norm(np.diff(p, axis=0), axis=1).max() > 0.5: continue
        q = p[:, :2] - p[:, :2].mean(0)
        u, sv, vt = np.linalg.svd(q, full_matrices=False)
        if sv[1] / max(sv[0], 1e-9) > 0.05 or np.sqrt(sv[1] ** 2 / chunk) > max_res: continue
        d = vt[0]; N.append([-d[1], d[0]]); C.append(p[:, :2].mean(0)); E.append([p[0, :2], p[-1, :2]])
    N, C, E = np.array(N), np.array(C), np.array(E); keep = np.ones(len(N), bool)
    for _ in range(iters):
        x = np.linalg.lstsq(N[keep], (N[keep] * C[keep]).sum(1), rcond=None)[0]
        res = np.abs((N * (x - C)).sum(1)); keep = res < max(0.05, np.percentile(res[keep], 80))
    return x, C[keep], E[keep], float(np.median(res[keep]))

def fig_scanner(st_full, st_zoom, out_png, out_pdf, n_lines=400, seed=0):
    """(a) plan view of a station: accepted scan-line runs, their planes towards the scanner, scan-line estimate vs the
    old floor-centre fallback; (b) close-up of a station with a tripod disc: floor points, disc, scan-line estimate."""
    rng = np.random.RandomState(seed)
    # FIG-006: printed width in the two-column layout (0.8 x 522 pt), all text 8 pt
    fig, ax = plt.subplots(1, 2, figsize=(0.8 * 522 / 72, 2.95), gridspec_kw=dict(width_ratios=[1.45, 1]))
    info = {}
    # (a)
    name, xyz = st_full
    fz, cz = floor_ceiling(xyz[:, 2])
    x, Ck, Ek, med = scanline_runs(xyz)
    sub = xyz[rng.choice(len(xyz), min(400000, len(xyz)), replace=False)]
    sub = sub[(sub[:, 2] > fz + 0.3) & (sub[:, 2] < cz - 0.2)]             # walls and furniture, not floor / ceiling
    a = ax[0]; a.scatter(sub[:, 0], sub[:, 1], s=0.05, c="0.70", lw=0, rasterized=True)
    pick = rng.choice(len(Ck), min(n_lines, len(Ck)), replace=False)
    for k in pick:
        a.plot([Ck[k, 0], x[0]], [Ck[k, 1], x[1]], color="#2a78d6", lw=0.25, alpha=0.25)
        a.plot(Ek[k, :, 0], Ek[k, :, 1], color="#2a78d6", lw=0.9)
    old, how_old = scanner_position(xyz, fz, 1.6)
    a.plot(*x, marker="*", ms=11, color="#e34948", mec="k", mew=0.4, ls="", label="scan-line estimate")
    a.plot(old[0], old[1], marker="X", ms=8, color="#eda100", mec="k", mew=0.4, ls="", label="old estimate: " + ("floor centre" if "floor centre" in how_old else "tripod disc"))
    dist = float(np.hypot(*(x - old[:2])))
    a.annotate(f"{dist:.2f} m", xy=(old[0], old[1]), xytext=(9, -3), textcoords="offset points", fontsize=8)
    a.set_aspect("equal"); a.set_title(f"(a) {name}: {len(Ck):,} scan-line runs (blue), median residual {med * 1000:.0f} mm", fontsize=8)
    a.set_xlabel("x (m)"); a.set_ylabel("y (m)"); a.legend(loc="lower left", fontsize=8, frameon=True, framealpha=0.9)
    info["full"] = dict(station=name, runs=int(len(Ck)), median_residual_m=round(med, 4), estimate=[round(float(v), 3) for v in x],
                        floor_centre=[round(float(v), 3) for v in old[:2]], distance_m=round(dist, 3), how_old=how_old)
    # (b)
    name, xyz = st_zoom
    fz, cz = floor_ceiling(xyz[:, 2])
    x, Ck, Ek, med = scanline_runs(xyz)
    fm = np.abs(xyz[:, 2] - fz) < 0.03
    disc = estimate_scanner_xy(xyz, np.where(fm, 12, -1), 12, 0.02)
    R = 1.3; f = xyz[fm]; f = f[(np.abs(f[:, 0] - x[0]) < R) & (np.abs(f[:, 1] - x[1]) < R)]
    a = ax[1]; a.scatter(f[:, 0], f[:, 1], s=0.08, c="0.35", lw=0, rasterized=True)
    if disc is not None:
        a.add_patch(Circle((disc[0], disc[1]), disc[2], fill=False, ec="#1baf7a", lw=1.2, ls="--"))
        a.plot(disc[0], disc[1], marker="+", ms=10, color="#1baf7a", mew=1.4, ls="", label=f"centre of tripod disc (r = {disc[2]:.2f} m)")
        dd = float(np.hypot(x[0] - disc[0], x[1] - disc[1]))
    else:
        dd = float("nan")
    a.plot(*x, marker="*", ms=11, color="#e34948", mec="k", mew=0.4, ls="", label=f"scan-line estimate ({dd * 100:.1f} cm off)")
    a.set_xlim(x[0] - R, x[0] + R); a.set_ylim(x[1] - R, x[1] + R); a.set_aspect("equal")
    a.set_title(f"(b) {name}: floor points around the scanner", fontsize=8); a.set_xlabel("x (m)"); a.set_ylabel("y (m)")
    a.legend(loc="lower left", fontsize=8, frameon=True, framealpha=0.9)
    info["zoom"] = dict(station=name, estimate=[round(float(v), 3) for v in x], disc=None if disc is None else [round(float(v), 3) for v in disc[:3]],
                        distance_m=round(dd, 4))
    plt.tight_layout(); plt.savefig(out_png, dpi=250); plt.savefig(out_pdf, dpi=300); plt.show(); plt.close(fig)
    return info

# class colours for Fig. 3: small classes saturated, everything else muted
PAL = {i: c for i, c in enumerate(["#9a9a9a", "#a7a39a", "#ecebe6", "#1baf7a", "#cbb89c", "#6250d6", "#e34948", "#cdbfae", "#d7b5a0",
                                   "#efe0a0", "#2a78d6", "#b0b0b0", "#c2c1bb", "#b5b5b5", "#eb6834", "#dcdbd6", "#a9c6de", "#bdbdbd"])}

def _hex(h): return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)], np.uint8)
LUT = np.stack([_hex(PAL[i]) for i in range(18)])

def object_frame(xyz, tree, ids, origin, radius=0.30):
    c0 = xyz[ids].mean(0)
    nb = tree.query_ball_point(c0, radius)
    q = xyz[nb] - xyz[nb].mean(0)
    w, V = np.linalg.eigh(q.T @ q); n = V[:, 0]
    if np.dot(origin - c0, n) < 0: n = -n                                   # normal towards the scanner
    up = np.array([0, 0, 1.0])
    u = np.cross(up, n) if abs(n[2]) < 0.9 else np.array([1.0, 0, 0]) - n * n[0]
    u /= np.linalg.norm(u); v = np.cross(n, u)
    return c0, u, v, n

def render_patch(xyz, tree, c0, u, v, n, half=0.22, px=0.003, depth=(-0.06, 0.15)):
    """Orthographic close-up looking along -n: for every pixel the point nearest to the viewer; small holes filled."""
    nb = np.array(tree.query_ball_point(c0, half * 1.6))
    d = xyz[nb] - c0
    a, b, h = d @ u, d @ v, d @ n
    m = (np.abs(a) < half) & (np.abs(b) < half) & (h > depth[0]) & (h < depth[1])
    nb, a, b, h = nb[m], a[m], b[m], h[m]
    W = int(2 * half / px); i = np.clip(((half - b) / px).astype(int), 0, W - 1); j = np.clip(((a + half) / px).astype(int), 0, W - 1)
    order = np.argsort(h)                                                  # nearest to the viewer written last
    idx = np.full((W, W), -1, np.int64); idx[i[order], j[order]] = nb[order]
    empty = idx < 0
    if empty.any() and (~empty).any():
        dist, (ii, jj) = ndimage.distance_transform_edt(empty, return_indices=True)
        fill = empty & (dist <= 2.0); idx[fill] = idx[ii[fill], jj[fill]]
    return idx

def colourise(idx, values, lut=None):
    img = np.full(idx.shape + (3,), 255, np.uint8); ok = idx >= 0
    img[ok] = values[idx[ok]] if lut is None else lut[values[idx[ok]]]
    return img

def fig_qualitative(rows, out_png, out_pdf):
    """rows: list of (title, rgb_patch, gt_patch, base_patch, hyb_patch)."""
    # FIG-005: 252 pt wide (one column of the two-column layout), all text 8 pt (= caption size)
    net = CFG["base_name"].split(",")[0]
    cols = ["colour\n(RGB)", "ground\ntruth", net + "\nalone", net + " +\n2D module"]
    fig, ax = plt.subplots(len(rows), 4, figsize=(252 / 72, 0.86 * len(rows) + 0.95), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01, wspace=0.02, hspace=0.02)
    ax = np.atleast_2d(ax)
    for r, (title, *ims) in enumerate(rows):
        for k, im in enumerate(ims):
            a = ax[r, k]; a.imshow(im, interpolation="nearest"); a.set_xticks([]); a.set_yticks([])
            for sp in a.spines.values(): sp.set_linewidth(0.4); sp.set_color("0.6")
            if r == 0: a.set_title(cols[k], fontsize=8)
        ax[r, 0].set_ylabel(title, fontsize=8)
    used = [3, 10, 14, 9, 15, 2, 4, 17]
    handles = [Patch(facecolor=PAL[c], edgecolor="0.5", lw=0.3, label=CLASS_NAMES[c]) for c in used]
    fig.legend(handles=handles, loc="outside lower center", ncol=4, fontsize=8, frameon=False, handlelength=1.0, columnspacing=0.8, handletextpad=0.4)
    plt.savefig(out_png, dpi=250); plt.savefig(out_pdf, dpi=300); plt.show(); plt.close(fig)


In [ ]:
# ---- Fig. 2 (scanner position) and Fig. 3 (qualitative validation) ----
FIGINFO = {}
# Fig. 2: (a) a test station without tripod disc, (b) a validation station with tripod disc - raw files, scan order kept
a = np.load(raw_file("test_area", CFG["fig2_full"])); xyz_full = (a[:, :3] - OFFSET).astype(np.float64); del a
a = np.load(raw_file("Gold", CFG["fig2_zoom"])); xyz_zoom = (a[:, :3] - OFFSET).astype(np.float64); del a
FIGINFO["fig2"] = fig_scanner((CFG["fig2_full"], xyz_full), (CFG["fig2_zoom"], xyz_zoom),
                              os.path.join(OUT, "fig_scanner_position.png"), os.path.join(OUT, "fig_scanner_position.pdf"))
print("Fig. 2:", json.dumps(FIGINFO["fig2"]))
del xyz_full, xyz_zoom

# Fig. 3: close-ups of validation objects: colour | ground truth | 3D network alone | 3D network + 2D module (selected setting)
if B1:
    hyb = {s: hybrid_module(base3d[s], PDv[s], PNv[s], CFG["fused_votes"], CFG["fused_cap"], STN[s]["xyz"], CFG["train"], True, damper=True)
           for s in CFG["val"]}
    m = metrics(sum(confusion(STN[s]["lab"].astype(np.int64), hyb[s]) for s in CFG["val"]))
    print("labels used in Fig. 3 - hybrid:", m["mIoU"], m["small_mIoU"], "(must equal the selected protocol-A line)")
    rows = []; FIGINFO["fig3"] = []
    for s in sorted(set(o[0] for o in CFG["fig3_objects"]) & set(STN) & set(base3d)):     # smoke test: only the stations that ran
        a = np.load(raw_file("Gold", s)); rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8); del a
        xyz = STN[s]["xyz"].astype(np.float64); tree = cKDTree(xyz); lab = STN[s]["lab"].astype(np.int64)
        origin, _ = scanner_position_v4(xyz, STN[s]["floor"], CFG["scanner_height_m"])
        for s2, c, title in CFG["fig3_objects"]:
            if s2 != s: continue
            objs = [o for cc, o in STN[s]["gobj"] if cc == c]
            if not objs: print("no", CLASS_NAMES[c], "object in", s); continue
            o = max(objs, key=len)
            c0, u, v, n = object_frame(xyz, tree, o, origin)
            idx = render_patch(xyz, tree, c0, u, v, n, half=CFG["fig3_half"], px=CFG["fig3_px"])
            ok = idx >= 0
            share = {k: round(float((L[o] == c).mean()), 3) for k, L in (("ptv3", base3d[s]), ("hybrid", hyb[s]))}
            FIGINFO["fig3"].append(dict(station=s, cls=CLASS_NAMES[c], points=int(len(o)), centre=[round(float(t), 3) for t in c0], recall=share))
            rows.append((f"{title}\n({s})", colourise(idx, rgb), colourise(idx, lab, LUT), colourise(idx, base3d[s], LUT), colourise(idx, hyb[s], LUT)))
        del xyz, tree, rgb
    order = [t for _, _, t in CFG["fig3_objects"]]
    rows.sort(key=lambda r: order.index(r[0].split("\n")[0]))
    fig_qualitative(rows, os.path.join(OUT, "fig_qualitative_val.png"), os.path.join(OUT, "fig_qualitative_val.pdf"))
    print("Fig. 3:", json.dumps(FIGINFO["fig3"]))
SUMMARY["figures"] = FIGINFO
with open(os.path.join(OUT, "summary.json"), "w") as fh: json.dump(SUMMARY, fh, indent=1)
print("\nsaved:", sorted(os.listdir(OUT)))


Record as **FIG-006** in `docs/experiment_log.md`: check that the protocol-A table reproduces HYB-007 (75.43 / 72.52). Download `FIG-006b_figs_8pt_printed_width/fig_scanner_position.pdf` and, from `FIG-003_data_pipeline_review/`, `fig_data_station.pdf` and `fig_pipeline_real.pdf` into `paper_aic/figures/` (not fig_review_effort: that one is drawn locally; not fig_qualitative_val: FIG-005 is used). Figures are drawn at their printed width in the two-column layout with 8 pt text.

In [ ]:
import types; detect = types.FunctionType(detect.__code__, dict(globals()), "detect")
# ---- FIG-003: data figure, pipeline with real images, review-effort curve (appended to notebook 16) ----
# Uses STN, base3d, hyb, CFG, render_view, detect, segment, merge_dets, features, SET_D, SET_N from the cells above.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
OUT3 = os.path.join(os.path.dirname(OUT), "FIG-003_data_pipeline_review")
os.makedirs(OUT3, exist_ok=True)
INFO3 = {}
SW, OU, DA = 14, 10, 3


def predict_props(train_st, test_st, parts):
    """Per-proposal class probabilities of the 2D classifier (same model and features as train_predict)."""
    train = [it for t in train_st for it in STN[t]["props"] + STN[t]["gtc"]]
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=CFG["C"], class_weight="balanced", max_iter=5000, random_state=CFG["seed"]))
    allt = [it for s in test_st for it in STN[s]["props"]]
    Xtr, Xte = features(train, allt, parts)
    clf.fit(Xtr, np.array([it["y"] for it in train]))
    pr = clf.predict_proba(Xte) if len(allt) else np.zeros((0, len(clf.classes_)))
    out, i0 = {}, 0
    for s in test_st:
        n = len(STN[s]["props"]); out[s] = pr[i0:i0 + n]; i0 += n
    return out, list(clf.classes_)


def fused_scores(PD, PN, classes):
    """Damper from D; Switch / Outlet from (1 - w) D + w N, as in the module."""
    w = CFG["fused_w"]; ci = {c: classes.index(c) for c in (DA, OU, SW) if c in classes}
    sc = {c: np.zeros(len(PD)) for c in (DA, OU, SW)}
    if DA in ci: sc[DA] = PD[:, ci[DA]]
    for c in (OU, SW):
        if c in ci: sc[c] = (1 - w) * PD[:, ci[c]] + w * PN[:, ci[c]]
    return sc


def candidates(S, r=None):
    """3D candidates a person would review: proposals whose centroids lie within merge_m are one candidate."""
    r = CFG["merge_m"] if r is None else r
    if not S["props"]: return np.zeros(0, int), []
    C = np.array([S["xyz"][p_["ids"]].mean(0) for p_ in S["props"]])
    comp = cluster_centroids(C, r)
    groups = [np.flatnonzero(comp == k) for k in range(comp.max() + 1)]
    return comp, groups


# ======================= (1) review effort, leave-one-station-out on the seven labelled stations =======================
ALL = [s for s in CFG["train"] + CFG["val"] if s in STN]
KMAX = 150
curves = {c: {"classifier": np.zeros(KMAX + 1), "detector": np.zeros(KMAX + 1)} for c in (SW, OU, DA)}
nobj = {c: 0 for c in (SW, OU, DA)}
per_station = {}
PHR = {SW: ["light switch"], OU: ["electrical outlet", "power socket"], DA: ["smoke detector"]}
for t in ALL:
    S = STN[t]; rest = [s for s in ALL if s != t]
    PD, cls = predict_props(rest, [t], SET_D); PN, cls2 = predict_props(rest, [t], SET_N)
    assert cls == cls2
    sc = fused_scores(PD[t], PN[t], cls)
    comp, groups = candidates(S)
    gd = np.array([p_["gd"] for p_ in S["props"]]) if S["props"] else np.zeros((0, P + 1))
    pts = [np.unique(np.concatenate([S["props"][i]["ids"] for i in g])) for g in groups]
    per_station[t] = dict(proposals=len(S["props"]), candidates=len(groups), seconds=S["seconds"])
    for c in (SW, OU, DA):
        objs = [o for cc, o in S["gobj"] if cc == c]
        nobj[c] += len(objs)
        if not objs or not groups: continue
        # candidate k finds object o if it covers at least 30 % of the object's points (the proposal-recall rule)
        hit = np.array([[np.isin(o, q, assume_unique=False).mean() >= 0.3 for o in objs] for q in pts])
        for name, score in (("classifier", np.array([sc[c][g].max() for g in groups])),
                            ("detector", np.array([gd[g][:, [CFG["phrases"].index(ph) for ph in PHR[c]]].max() for g in groups]))):
            order = np.argsort(-score, kind="stable")
            found = np.zeros(len(objs), bool)
            for K in range(1, KMAX + 1):
                if K <= len(order): found |= hit[order[K - 1]]
                curves[c][name][K] += found.sum()
    print(f"{t}: {len(S['props'])} proposals -> {len(groups)} candidates; objects " +
          ", ".join(f"{CLASS_NAMES[c]} {sum(cc == c for cc, _ in S['gobj'])}" for c in (SW, OU, DA)))

REV = {}
for c in (SW, OU, DA):
    REV[CLASS_NAMES[c]] = dict(objects=nobj[c], **{name: {f"top{K}": int(curves[c][name][K]) for K in (1, 3, 5, 10, 20, 30, 50, 100, 150)}
                                                   for name in ("classifier", "detector")})
    for name in ("classifier", "detector"):
        full = np.flatnonzero(curves[c][name] >= nobj[c]) if nobj[c] else []
        REV[CLASS_NAMES[c]][name]["K_all_found"] = int(full[0]) if len(full) else None
INFO3["review_effort_LOSO"] = REV; INFO3["stations"] = per_station
print("review effort (objects found among the top-K candidates per station, pooled over the 7 held-out stations):")
print(json.dumps(REV, indent=1))

kx = [v[n]["K_all_found"] for v in REV.values() for n in ("classifier", "detector") if v["objects"]]
XMAX = min(KMAX, max([k for k in kx if k is not None] or [KMAX]) + 5) if all(k is not None for k in kx) else KMAX
fig, axs = plt.subplots(1, 3, figsize=(6.8, 2.1), sharey=True)
for a, c in zip(axs, (SW, OU, DA)):
    K = np.arange(0, KMAX + 1)
    for name, st, lab_ in (("classifier", "-", "2D classifier score"), ("detector", "--", "detector phrase score")):
        y = curves[c][name] / max(nobj[c], 1) * 100; y[0] = 0
        a.step(K, y, where="post", ls=st, lw=1.3, color="#eb6834" if name == "classifier" else "0.35", label=lab_)
    a.set_title(f"{CLASS_NAMES[c]} ({nobj[c]} objects)", fontsize=8); a.set_xlabel("candidates reviewed per station")
    a.set_xlim(0, XMAX); a.set_ylim(0, 102); a.grid(lw=0.3, color="0.85")
axs[0].set_ylabel("objects found (%)"); axs[0].legend(loc="lower right", fontsize=6.5, frameon=False)
plt.tight_layout(); plt.savefig(os.path.join(OUT3, "fig_review_effort.png"), dpi=250, bbox_inches="tight", pad_inches=0.02); plt.savefig(os.path.join(OUT3, "fig_review_effort.pdf"), bbox_inches="tight", pad_inches=0.02); plt.show(); plt.close(fig)

# ======================= (2) data figure and (3) pipeline with real images: S163, view towards the switch =======================
s = CFG["val"][0] if CFG["val"] else ALL[0]
S = STN[s]
a = np.load(raw_file("Gold", s)); rgb = np.clip(a[:, 3:6], 0, 255).astype(np.uint8); inten = a[:, 6].astype(np.float32); del a
xyz = S["xyz"].astype(np.float64); lab = S["lab"].astype(np.int64)
origin, _ = scanner_position_v4(xyz, S["floor"], CFG["scanner_height_m"])
sw = [o for cc, o in S["gobj"] if cc == SW]
o = max(sw, key=len) if sw else max([o for _, o in S["gobj"]], key=len)
q = xyz[o].mean(0) - origin; q /= np.linalg.norm(q)
vi = int(np.argmax([view_dirs(y_, p_)[0] @ q for y_, p_ in CFG["views"]]))          # the module's own view that looks most directly at it
yw, pt = CFG["views"][vi]
img, I, IDX, proj = render_view(xyz, rgb, inten, origin, yw, pt, size=CFG["size"], fov_deg=CFG["fov"])
g = stretch(I); pics = {"rgb": Image.fromarray(img), "int": Image.fromarray(np.dstack([g, g, g]))}
on = np.isin(IDX, o)
ys, xs = np.nonzero(on)
bx = (xs.min(), ys.min(), xs.max() + 1, ys.max() + 1) if len(xs) else (600, 600, 680, 680)
INFO3["view"] = dict(station=s, view_index=vi, yaw=yw, pitch=pt, object_class=CLASS_NAMES[lab[o[0]]], object_points=int(len(o)),
                     station_points=int(S["n"]), object_pixels=int(on.sum()), bbox=[int(v) for v in bx])

def labimg(IDX, values, lut=LUT):
    out = np.full(IDX.shape + (3,), 255, np.uint8); ok = IDX >= 0; out[ok] = lut[values[IDX[ok]]]; return out

def frame(a):
    a.set_xticks([]); a.set_yticks([])
    for sp in a.spines.values(): sp.set_linewidth(0.4); sp.set_color("0.6")

def box(a, b, col="#e34948", lw=1.0):
    a.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=col, lw=lw))

# (2) data figure: colour | intensity | ground truth | PTv3 alone, whole view, the object marked
pad = 18; mark = (bx[0] - pad, bx[1] - pad, bx[2] + pad, bx[3] + pad)
gt_img = labimg(IDX, lab); b3 = base3d.get(s)
panels = [("(a) colour (RGB)", img), ("(b) laser intensity", np.dstack([g, g, g])), ("(c) ground truth", gt_img)]
if b3 is not None: panels.append(("(d) " + CFG["base_name"].split(",")[0] + " alone", labimg(IDX, b3)))
_H = 154.0   # FIG-006: printed width (0.9 x 522 pt), text 8 pt, fixed margins in pt: title 15, legend 22
fig, axs = plt.subplots(1, len(panels), figsize=(0.9 * 522 / 72, _H / 72))
for a, (t_, im) in zip(np.atleast_1d(axs), panels):
    a.imshow(im, interpolation="nearest"); frame(a); a.set_title(t_, fontsize=8); box(a, mark)
_vis = lab[IDX[IDX >= 0]]
present = [c for c in range(18) if (len(_vis) and (_vis == c).mean() > 0.004) or c == lab[o[0]]]
handles = [Patch(facecolor=PAL[c], edgecolor="0.5", lw=0.3, label=CLASS_NAMES[c]) for c in present]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=min(len(handles), 8), fontsize=8, frameon=False, handlelength=1.0, columnspacing=0.8)
plt.subplots_adjust(left=0.004, right=0.996, top=1 - 15 / _H, bottom=22 / _H, wspace=0.025); plt.savefig(os.path.join(OUT3, "fig_data_station.png"), dpi=250); plt.savefig(os.path.join(OUT3, "fig_data_station.pdf"), dpi=300); plt.show(); plt.close(fig)
INFO3["data_fig_classes"] = [CLASS_NAMES[c] for c in present]

# (3) pipeline with real images, cropped around the object: boxes | SAM masks | classified 3D proposals | final labels
raw_d = [d for src in CFG["sources"] for d in detect(pics[src])]
dets = merge_dets(raw_d) if raw_d else []
masks = segment(pics["rgb"], [d[0] for d in dets]) if dets else []
PDs, cls = predict_props(CFG["train"], [s], SET_D); PNs, _ = predict_props(CFG["train"], [s], SET_N)
sc = fused_scores(PDs[s], PNs[s], cls)
ci = {c: i for i, c in enumerate(cls)}
pv = np.full(S["n"], -1, np.int64)                          # class of the proposals of this view, per point
best = np.zeros(S["n"])
nprop_view = 0
for j, p_ in enumerate(S["props"]):
    if p_["view"] != vi: continue
    nprop_view += 1
    probs = {c: sc[c][j] for c in (DA, OU, SW)}
    pbg = PDs[s][j, ci[-1]] if -1 in ci else 0.0
    c_best = max(probs, key=probs.get)
    c_ = c_best if probs[c_best] >= pbg else 17               # background proposals drawn in the clutter grey
    upd = probs[c_best] > best[p_["ids"]]
    pv[p_["ids"][upd]] = c_; best[p_["ids"][upd]] = probs[c_best]
c0x, c0y = (bx[0] + bx[2]) // 2, (bx[1] + bx[3]) // 2; H = 170
cr = (max(c0x - H, 0), max(c0y - H, 0), min(c0x + H, CFG["size"]), min(c0y + H, CFG["size"]))
sl = (slice(cr[1], cr[3]), slice(cr[0], cr[2]))
in_crop = lambda b: b[2] > cr[0] and b[0] < cr[2] and b[3] > cr[1] and b[1] < cr[3]
ov = img.astype(float).copy(); rs = np.random.RandomState(1)
for m in masks:
    col = rs.randint(40, 230, 3); ov[m] = 0.45 * ov[m] + 0.55 * col
prop_img = img.copy(); okp = (IDX >= 0); okp[okp] = pv[IDX[okp]] >= 0
prop_img = (0.35 * prop_img + 0.65 * 255).astype(np.uint8); prop_img[okp] = LUT[pv[IDX[okp]]]
fin = hyb[s] if (isinstance(hyb, dict) and s in hyb) else (b3 if b3 is not None else lab)
fin_img = labimg(IDX, fin)
pp = [("(a) detector boxes", img, True), ("(b) SAM masks", ov.astype(np.uint8), False),
      ("(c) proposals in 3D,\nclassified", prop_img, False), ("(d) after voting\nand size check", fin_img, False)]
_H = 163.0   # FIG-006: printed width (0.9 x 522 pt), text 8 pt, fixed margins in pt: two-line title 24, legend 22
fig, axs = plt.subplots(1, 4, figsize=(0.9 * 522 / 72, _H / 72))
for a, (t_, im, draw) in zip(axs, pp):
    a.imshow(im[sl], interpolation="nearest"); frame(a); a.set_title(t_, fontsize=8)
    if draw:
        for d in dets:
            b = d[0]
            if in_crop(b): box(a, (b[0] - cr[0], b[1] - cr[1], b[2] - cr[0], b[3] - cr[1]), "#2a78d6", 0.6)
# legend: only the classes that actually appear in panels (c) and (d) of the crop, plus the box colour of (a)
_pc = pv[IDX[sl][IDX[sl] >= 0]]; _fc = fin[IDX[sl][IDX[sl] >= 0]]
used = [c for c in [SW, OU, DA, 17] + sorted(set(range(18)) - {SW, OU, DA, 17}) if (_pc == c).mean() > 0.002 or (_fc == c).mean() > 0.002]
handles = [Patch(facecolor="none", edgecolor="#2a78d6", lw=0.6, label="detector box")] + [Patch(facecolor=PAL[c], edgecolor="0.5", lw=0.3, label=(CLASS_NAMES[c] if c != 17 else "background / clutter")) for c in used]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(handles), fontsize=8, frameon=False, handlelength=1.0, columnspacing=0.8)
plt.subplots_adjust(left=0.004, right=0.996, top=1 - 24 / _H, bottom=22 / _H, wspace=0.025); plt.savefig(os.path.join(OUT3, "fig_pipeline_real.png"), dpi=250); plt.savefig(os.path.join(OUT3, "fig_pipeline_real.pdf"), dpi=300); plt.show(); plt.close(fig)
INFO3["pipeline_fig"] = dict(view_detections_merged=len(dets), boxes_in_crop=int(sum(in_crop(d[0]) for d in dets)),
                             proposals_from_view=nprop_view, crop=[int(v) for v in cr],
                             object_recall_final=round(float((fin[o] == lab[o[0]]).mean()), 3),
                             object_recall_3d=round(float((b3[o] == lab[o[0]]).mean()), 3) if b3 is not None else None)
print("FIG-003:", json.dumps({k: v for k, v in INFO3.items() if k != "review_effort_LOSO"}))
with open(os.path.join(OUT3, "summary_fig003.json"), "w") as fh: json.dump(INFO3, fh, indent=1)
print("saved:", sorted(os.listdir(OUT3)))


In [ ]:
print("CURVES", json.dumps({CLASS_NAMES[c]: dict(objects=int(nobj[c]), **{n: [int(v) for v in curves[c][n][:61]] for n in curves[c]}) for c in curves}))